# OpenPI Drone VLA - Training Pipeline 

**Alur Utama:**
1. Setup & Verifikasi Dataset
2. Install Dependencies
3. Copy & Extract Dataset
4. Patching & Konfigurasi
5. Training & Monitoring
6. Testing & Inference

## 1️. Setup & Verifikasi Path Dataset

Memeriksa ketersediaan dataset di Kaggle environment dan mempersiapkan path untuk semua komponen.

In [1]:
import os, subprocess, shutil, sys

# Nama dataset kamu di Kaggle (URL Slug)
DATASET_NAME = 'dronepivla' 
# Kaggle biasanya menaruh data langsung di /kaggle/input/[nama-dataset]
KAGGLE_BASE = f'/kaggle/input//datasets/sjankaczar/{DATASET_NAME}'

# Kita siapkan path untuk versi folder (karena Kaggle ekstrak otomatis)
# dan versi zip (jika ternyata tidak diekstrak)
OPENPI_DIR     = f'{KAGGLE_BASE}/openpi/openpi'
DATASET_DIR    = f'{KAGGLE_BASE}/drone_roblox/drone_roblox'
NORMSTATS_DIR  = f'{KAGGLE_BASE}/norm_stats'

# Alternatif jika masih berupa .zip
OPENPI_ZIP     = f'{KAGGLE_BASE}/openpi.zip'
DATASET_ZIP    = f'{KAGGLE_BASE}/drone_roblox.zip'
NORMSTATS_ZIP  = f'{KAGGLE_BASE}/norm_stats.zip'

print('✅ Lingkungan Kaggle Terdeteksi. Memeriksa file...')

# Verifikasi & Penentuan Path yang Benar
def check_path(dir_path, zip_path):
    if os.path.isdir(dir_path): return dir_path
    if os.path.exists(zip_path): return zip_path
    return None

FINAL_OPENPI    = check_path(OPENPI_DIR, OPENPI_ZIP)
FINAL_DATASET   = check_path(DATASET_DIR, DATASET_ZIP)
FINAL_NORMSTATS = check_path(NORMSTATS_DIR, NORMSTATS_ZIP)

missing = False
for name, p in [("OpenPI", FINAL_OPENPI), ("Dataset", FINAL_DATASET), ("NormStats", FINAL_NORMSTATS)]:
    if p:
        print(f"  ✅ {name} Ditemukan di: {p}")
    else:
        print(f"  ❌ {name} TIDAK DITEMUKAN!")
        missing = True

if missing:
    print("\n⚠️ PERINGATAN: File tidak ditemukan di path default.")
    print("Mencoba melacak struktur folder di /kaggle/input/ ...")
    # Menampilkan isi folder untuk memudahkan debugging
    for root, dirs, files in os.walk('/kaggle/input/'):
        level = root.replace('/kaggle/input/', '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
else:
    print("\n🚀 Semua file siap untuk diproses ke /kaggle/working/")



✅ Lingkungan Kaggle Terdeteksi. Memeriksa file...
  ✅ OpenPI Ditemukan di: /kaggle/input//datasets/sjankaczar/dronepivla/openpi/openpi
  ✅ Dataset Ditemukan di: /kaggle/input//datasets/sjankaczar/dronepivla/drone_roblox/drone_roblox
  ✅ NormStats Ditemukan di: /kaggle/input//datasets/sjankaczar/dronepivla/norm_stats

🚀 Semua file siap untuk diproses ke /kaggle/working/


## 2️. Install Dependencies

Menginstal semua library yang diperlukan: PyTorch, JAX, Flax, Transformers, dan utilities lainnya. **PENTING:** Setelah cell ini selesai, klik menu "Run" → "Restart Kernel"!

In [2]:
%%time
# ^-- Tetap di baris pertama tanpa ada komentar di atasnya!

import subprocess, sys

def pip_install(*args):
    """Run pip install with given arguments."""
    # Kita tambahkan --index-url untuk memastikan Torch mendeteksi CUDA T4
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-warn-conflicts', *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  Warning during install: {result.stderr[-500:]}')
    return result.returncode

# Step 0: Pembersihan Total (Kaggle sering punya library sisa yang bentrok)
print('🧹 Step 0: Nuking existing JAX, Numpy, and Torch to prevent version mismatch...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'jax', 'jaxlib', 'numpy', 'torch', 'torchaudio', 'torchvision', '-y'], capture_output=True)

print('📦 Step 1: Installing Numpy 1.26.4 (CRITICAL: Fixes the umath error)...')
# OpenPI & Transformers wajib pakai Numpy 1.x!
pip_install('numpy==1.26.4')

print('📦 Step 2: Installing Torch & Core ML libraries...')
# Kita gunakan 2.4.0 karena sangat stabil untuk driver T4 Kaggle (2.7.x belum stabil/resmi)
pip_install(
    'torch==2.4.0', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121',
    'safetensors>=0.4.0',
    'einops>=0.8.0',
    'wandb>=0.19.1',
)

print('📦 Step 3: Installing JAX + Flax ecosystem (0.5.3)...')
pip_install(
    'jax[cuda12]==0.5.3',
    'jaxlib==0.5.3',
    'flax==0.10.2',
    'orbax-checkpoint==0.11.13',
    'chex',
    'equinox>=0.11.8',
    'ml-collections==1.0.0',
    'jaxtyping==0.2.36',
    'ml-dtypes==0.4.1',
)

print('📦 Step 4: Installing data libraries...')
pip_install(
    'lerobot@git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5',
    'datasets',
    'huggingface-hub',
)

print('📦 Step 5: Installing all utility libraries (FULL LIST)...')
pip_install(
    'tyro>=0.9.5',
    'beartype==0.19.0', # Kunci versi ini agar tidak bentrok
    'numpydantic>=1.6.6',
    'sentencepiece>=0.2.0',
    'tqdm-loggable>=0.2',
    'humanize',
    'filelock>=3.16.1',
    'treescope>=0.1.7',
    'augmax>=0.3.4',
    'dm-tree>=0.1.8',
    'flatbuffers>=24.3.25',
    'fsspec==2023.10.0', # Menghindari konflik fsspec di Kaggle
    'gcsfs',
    'imageio>=2.36.1',
    'opencv-python-headless',
    'pillow>=11.0.0',
    'rich>=14.0.0',
    'polars>=1.30.0',
    'transformers==4.53.2',
    'gym-aloha>=0.1.1',
)

# Editable Install OpenPI dipindahkan ke Cell 3 sesuai alur Kaggle terbaru kita.

print('\n✅ All dependencies installed (Full List)!')
print('⚠️  WAJIB: Klik menu "Run" (panel atas Kaggle) -> "Restart Kernel"!')



🧹 Step 0: Nuking existing JAX, Numpy, and Torch to prevent version mismatch...
📦 Step 1: Installing Numpy 1.26.4 (CRITICAL: Fixes the umath error)...
📦 Step 2: Installing Torch & Core ML libraries...
📦 Step 3: Installing JAX + Flax ecosystem (0.5.3)...
📦 Step 4: Installing data libraries...
📦 Step 5: Installing all utility libraries (FULL LIST)...

✅ All dependencies installed (Full List)!
⚠️  WAJIB: Klik menu "Run" (panel atas Kaggle) -> "Restart Kernel"!
CPU times: user 17.3 ms, sys: 4.57 ms, total: 21.8 ms
Wall time: 6min 43s


## 3️. Copy & Extract Dataset ke Working Directory

Menyiapkan OpenPI, Dataset LeRobot, dan Norm Stats dengan editable install untuk OpenPI.

In [3]:
import os, shutil, glob, subprocess, sys, zipfile

# PATH TUJUAN UTAMA
HOME_DIR = os.path.expanduser("~")
TARGET_DATA_DIR = os.path.join(HOME_DIR, ".cache/huggingface/lerobot/rafli/drone_roblox")
WORK_DIR = '/kaggle/working/openpi'

def smart_copy_or_extract(source, target_parent, target_name):
    """Menangani folder atau zip dari input ke working directory."""
    target_path = os.path.join(target_parent, target_name)
    if os.path.exists(target_path): shutil.rmtree(target_path)
    
    if os.path.isdir(source):
        print(f"📁 Menyalin folder {os.path.basename(source)}...")
        shutil.copytree(source, target_path)
    elif zipfile.is_zipfile(source):
        print(f"📦 Mengekstrak zip {os.path.basename(source)}...")
        with zipfile.ZipFile(source, 'r') as z:
            z.extractall(target_parent)
    else:
        print(f"❌ ERROR: Sumber tidak valid: {source}")

# 1. Siapkan OpenPI (Akan ada di /kaggle/working/openpi)
print("🚀 1/3: Menyiapkan OpenPI...")
smart_copy_or_extract(FINAL_OPENPI, '/kaggle/working', 'openpi')

# --- LANGKAH PENTING: Editable Install ---
print("📦 Menyambungkan kode OpenPI ke sistem Python...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', WORK_DIR, '--no-deps'])

# 2. Siapkan Dataset LeRobot (Di folder Cache)
print("\n🚀 2/3: Menyiapkan Dataset LeRobot...")
temp_data_path = '/kaggle/working/temp_dataset'
smart_copy_or_extract(FINAL_DATASET, '/kaggle/working', 'temp_dataset')

# Cari di mana 'meta/info.json' berada
matches = glob.glob(os.path.join(temp_data_path, "**/meta/info.json"), recursive=True)
if matches:
    actual_root = os.path.dirname(os.path.dirname(matches[0]))
    os.makedirs(os.path.dirname(TARGET_DATA_DIR), exist_ok=True)
    if os.path.exists(TARGET_DATA_DIR): shutil.rmtree(TARGET_DATA_DIR)
    shutil.move(actual_root, TARGET_DATA_DIR)
    print(f"✅ Dataset siap di: {TARGET_DATA_DIR}")
else:
    print("❌ ERROR: info.json tidak ditemukan!")

# 3. Siapkan Norm Stats (Logika diperbaiki!)
print("\n🚀 3/3: Menyiapkan Norm Stats...")
# Tujuan mutlak: /kaggle/working/assets/pi0_drone_lite/rafli/drone_roblox/
FINAL_TARGET_NS = '/kaggle/working/assets/pi0_drone_lite/rafli/drone_roblox'
temp_ns_path = '/kaggle/working/temp_ns'

# Ekstrak/Copy dulu ke temp
smart_copy_or_extract(FINAL_NORMSTATS, '/kaggle/working', 'temp_ns')

# Cari di mana norm_stats.json bersembunyi
ns_matches = glob.glob(os.path.join(temp_ns_path, "**/norm_stats.json"), recursive=True)
if ns_matches:
    ns_root = os.path.dirname(ns_matches[0])
    print(f"✅ Norm stats ditemukan di: {ns_root}")
    if os.path.exists(FINAL_TARGET_NS): shutil.rmtree(FINAL_TARGET_NS)
    os.makedirs(os.path.dirname(FINAL_TARGET_NS), exist_ok=True)
    
    # Salin seluruh isi folder yang berisi norm_stats.json ke target
    shutil.copytree(ns_root, FINAL_TARGET_NS)
    print(f"🚀 Norm Stats siap di: {FINAL_TARGET_NS}")
    print(f"📊 File di dalam: {os.listdir(FINAL_TARGET_NS)}")
else:
    print(f"❌ ERROR: norm_stats.json tidak ditemukan di dalam {FINAL_NORMSTATS}!")

# Bersihkan sampah temp
for t in [temp_data_path, temp_ns_path]:
    if os.path.exists(t): shutil.rmtree(t)

print("\n✨ VERIFIKASI SELESAI. Silakan lanjut ke Cell Patching & Training.")



🚀 1/3: Menyiapkan OpenPI...
📁 Menyalin folder openpi...
📦 Menyambungkan kode OpenPI ke sistem Python...
Obtaining file:///kaggle/working/openpi
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openpi (pyproject.toml): started
  Building editable for openpi (pyproject.toml): finished with status 'done'
  Created wheel for openpi: filename=openpi-0.1.0-py3-none-any.whl size=20285 sha256=a5138a0a9f7f9350525c06b4583c4ee56fa

## 4️. Verifikasi & Patching

Memastikan semua dependencies terinstall dengan benar. Testing import dan GPU detection.

In [4]:
import os, torch, sys, shutil

# Kita asumsikan Cell 2 (Install) dan Restart Kernel sudah dilakukan!
import transformers, lerobot
HOME_DIR = os.path.expanduser("~")
TARGET_DATA_DIR = os.path.join(HOME_DIR, ".cache/huggingface/lerobot/rafli/drone_roblox")

# --- FUNGSI HELPER: PATCHING HALUS ---
def patch_recursive(src, dst):
    """Menyisipkan file satu per satu tanpa menghapus folder asli sistem."""
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            if not os.path.exists(d): os.makedirs(d)
            patch_recursive(s, d)
        else:
            shutil.copy2(s, d)

print("🛠️ Memulai proses patching file...")

# 1. Patch OpenPI (Fix datetime.UTC)
download_py = '/kaggle/working/openpi/src/openpi/shared/download.py'
if os.path.exists(download_py):
    with open(download_py, 'r') as f: 
        content = f.read().replace('datetime.UTC', 'datetime.timezone.utc')
    with open(download_py, 'w') as f: 
        f.write(content)
    print("✅ Patch 1/3: OpenPI download.py OK.")

# 2. Patch LeRobot (Fix TypeError: stack())
lerobot_lib_path = os.path.dirname(lerobot.__file__)
lerobot_ds_py = os.path.join(lerobot_lib_path, 'common/datasets/lerobot_dataset.py')
if os.path.exists(lerobot_ds_py):
    with open(lerobot_ds_py, 'r') as f: 
        content = f.read()
    replacements = [
        ('torch.stack(self.hf_dataset["timestamp"])', 'torch.stack(list(self.hf_dataset["timestamp"]))'),
        ('torch.stack(self.hf_dataset["episode_index"])', 'torch.stack(list(self.hf_dataset["episode_index"]))'),
        ('torch.stack(self.hf_dataset["index"])', 'torch.stack(list(self.hf_dataset["index"]))')
    ]
    for old, new in replacements: 
        content = content.replace(old, new)
    with open(lerobot_ds_py, 'w') as f: 
        f.write(content)
    print("✅ Patch 2/3: LeRobot dataset.py OK.")

# 3. Patch Transformers (OpenPI Custom Model Patch)
transformers_path = os.path.dirname(transformers.__file__)
patch_source = '/kaggle/working/openpi/src/openpi/models_pytorch/transformers_replace'
if os.path.exists(patch_source):
    patch_recursive(patch_source, transformers_path)
    print("✅ Patch 3/3: Transformers OpenPI-Files OK.")
else:
    print("❌ ERROR: Folder 'transformers_replace' tidak ditemukan!")

# 4. Verifikasi Final & Testing Import
print("-" * 50)
try:
    import numpy
    print(f"📊 Numpy version: {numpy.__version__} (Harus 1.26.x)")
    from transformers import GemmaForCausalLM
    print("🧪 Test Import GemmaForCausalLM: BERHASIL!")
except Exception as e:
    print(f"🧪 Test Import GemmaForCausalLM: GAGAL! ({e})")

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram:.1f} GB (Deteksi Berhasil)")
else:
    print("❌ GPU TIDAK TERDETEKSI! Pastikan Accelerator GPU T4 sudah menyala.")

if os.path.exists(os.path.join(TARGET_DATA_DIR, 'meta/info.json')):
    print("✅ Dataset Cache: DITEMUKAN!")
else:
    print(f"❌ Dataset Cache: TIDAK DITEMUKAN di {TARGET_DATA_DIR}!")




🛠️ Memulai proses patching file...
✅ Patch 1/3: OpenPI download.py OK.
✅ Patch 2/3: LeRobot dataset.py OK.
✅ Patch 3/3: Transformers OpenPI-Files OK.
--------------------------------------------------
📊 Numpy version: 2.0.2 (Harus 1.26.x)


2026-04-10 13:32:15.217485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775827935.422375      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775827935.485084      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775827935.975354      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775827935.975389      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775827935.975392      55 computation_placer.cc:177] computation placer alr

🧪 Test Import GemmaForCausalLM: BERHASIL!
✅ GPU: Tesla T4 | VRAM: 15.6 GB (Deteksi Berhasil)
✅ Dataset Cache: DITEMUKAN!


## 5️. Konfigurasi Training

Mengatur parameter training: batch size, jumlah steps, save interval, dll.

In [5]:
import os

# Di Kaggle, semua hasil output disimpan di /kaggle/working/
# Kamu bisa mendownload folder ini setelah training selesai.
CHECKPOINT_DIR = f'/kaggle/working/checkpoints'
CONFIG_NAME    = 'pi0_drone_lite'       # Nama config yang sudah kita registrasi
EXP_NAME       = 'drone_gimbal_v1'      # Nama eksperimen kamu
NUM_STEPS      = 3000                   # Total langkah training (3000 butuh ~2-3 jam di Kaggle)
BATCH_SIZE     = 2                      # 🚀 KAGLE POWER: Batch size 2 sangat aman di RAM 30GB!
SAVE_INTERVAL  = 500                    # Simpan checkpoint setiap 500 langkah
LOG_INTERVAL   = 10                     # Catat log setiap 10 langkah
USE_WANDB      = False                  # Set True jika kamu ingin pakai Weights & Biases

# ----- Advanced Options -----
MIXED_PRECISION = 'bfloat16'            # 'bfloat16' paling efisien untuk arsitektur Pi0 di T4

print('--- 🚁 Training Configuration (Kaggle Edition) ---')
print(f'Config Name:      {CONFIG_NAME}')
print(f'Experiment Name:  {EXP_NAME}')
print(f'Total Steps:      {NUM_STEPS}')
print(f'Batch Size:       {BATCH_SIZE} (RAM 30GB Kaggle OK!)')
print(f'Save Interval:    {SAVE_INTERVAL}')
print(f'Precision:        {MIXED_PRECISION}')
print(f'W&B logging:      {USE_WANDB}')

# Buat folder checkpoint
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'\n💾 Checkpoint lokal akan disimpan di: {CHECKPOINT_DIR}')


--- 🚁 Training Configuration (Kaggle Edition) ---
Config Name:      pi0_drone_lite
Experiment Name:  drone_gimbal_v1
Total Steps:      3000
Batch Size:       2 (RAM 30GB Kaggle OK!)
Save Interval:    500
Precision:        bfloat16
W&B logging:      False

💾 Checkpoint lokal akan disimpan di: /kaggle/working/checkpoints


## 6️. Patch LeRobot - Fix TypeError Stack

Memperbaiki issue TypeError di LeRobot dataset module dengan mengkonversi dataset iterator ke list sebelum torch.stack().

In [6]:
# ============================================================
# Cell 6: Patch LeRobot (Fix TypeError: stack() - KAGGLE)
# ============================================================
import os, lerobot

# Kita cari lokasi library lerobot secara dinamis agar 100% tepat
lerobot_lib_path = os.path.dirname(lerobot.__file__)
lerobot_file = os.path.join(lerobot_lib_path, 'common/datasets/lerobot_dataset.py')

print(f"🔍 Mencari file di: {lerobot_file}")

if os.path.exists(lerobot_file):
    with open(lerobot_file, 'r') as f:
        content = f.read()

    # Daftar perbaikan
    replacements = [
        ('torch.stack(self.hf_dataset["timestamp"])', 'torch.stack(list(self.hf_dataset["timestamp"]))'),
        ('torch.stack(self.hf_dataset["episode_index"])', 'torch.stack(list(self.hf_dataset["episode_index"]))'),
        ('torch.stack(self.hf_dataset["index"])', 'torch.stack(list(self.hf_dataset["index"]))'),
        ('torch.stack(self.hf_dataset["task_index"])', 'torch.stack(list(self.hf_dataset["task_index"]))'),
        ('torch.stack(self.hf_dataset.select(q_idx)[key])', 'torch.stack(list(self.hf_dataset.select(q_idx)[key]))')
    ]

    patched_count = 0
    for old, new in replacements:
        if old in content:
            content = content.replace(old, new)
            patched_count += 1
            print(f"✅ Patching: {old[:40]}...")

    if patched_count > 0:
        with open(lerobot_file, 'w') as f:
            f.write(content)
        print(f"\n🎉 Berhasil menerapkan {patched_count} patch! LeRobot siap digunakan.")
    else:
        print("\nℹ️  Target tidak ditemukan (mungkin sudah pernah di-patch sebelumnya).")
else:
    print(f"❌ ERROR: File LeRobot tidak ditemukan!")


🔍 Mencari file di: /usr/local/lib/python3.12/dist-packages/lerobot/common/datasets/lerobot_dataset.py
✅ Patching: torch.stack(self.hf_dataset.select(q_idx...

🎉 Berhasil menerapkan 1 patch! LeRobot siap digunakan.


## 7️. Pre-download PaliGemma Tokenizer

Download tokenizer model PaliGemma ke cache lokal untuk menghindari error saat pertama kali load model.

In [7]:
# ============================================================
# Cell Perbaikan: Pre-download PaliGemma Tokenizer (Sesi Baru)
# ============================================================
import os
import urllib.request

print("🛠️ Mempersiapkan PaliGemma Tokenizer secara manual...")

cache_dir = '/root/.cache/openpi/big_vision'
os.makedirs(cache_dir, exist_ok=True)

tokenizer_url = "https://storage.googleapis.com/big_vision/paligemma_tokenizer.model"
target_path = os.path.join(cache_dir, "paligemma_tokenizer.model")

if not os.path.exists(target_path):
    print(f"⬇️ Sedang mengunduh Tokenizer ke {target_path} (sekitar 4 MB)...")
    try:
        urllib.request.urlretrieve(tokenizer_url, target_path)
        print("✅ Unduhan Tokenizer BERHASIL!")
    except Exception as e:
        print(f"❌ ERROR saat mengunduh: {e}")
else:
    print("✅ Tokenizer sudah tersedia di direktori target.")

🛠️ Mempersiapkan PaliGemma Tokenizer secara manual...
⬇️ Sedang mengunduh Tokenizer ke /root/.cache/openpi/big_vision/paligemma_tokenizer.model (sekitar 4 MB)...
✅ Unduhan Tokenizer BERHASIL!


## 8️. Inject LoRA (Low-Rank Adaptation)

Memasang LoRA adapter ke semua lapisan linear model untuk mengurangi beban VRAM dan mempercepat training.

In [8]:
# ============================================================
# Cell Pamungkas: PROTOKOL LORA (Low-Rank Adaptation) untuk VLA
# ============================================================
import os, subprocess, re

print("📦 Mengunduh perlengkapan operasi LoRA (PEFT)...")
os.system("pip install peft -q")

script_path = '/kaggle/working/openpi/scripts/train_pytorch.py'

if os.path.exists(script_path):
    print("🧹 Melakukan Factory Reset pada skrip untuk membuang patch lama...")
    subprocess.run(["git", "checkout", "scripts/train_pytorch.py"], cwd="/kaggle/working/openpi", check=False)

    with open(script_path, 'r') as f:
        content = f.read()

    # Kode chip memori LoRA yang akan ditanamkan ke otak VLA
    lora_code = """
# --- PROTOKOL LORA ---
print("\\n" + "="*60)
print("🚀 MENGAKTIFKAN LORA (LOW-RANK ADAPTATION) PADA VLA...")
try:
    from peft import get_peft_model, LoraConfig
    lora_config = LoraConfig(
        r=8, 
        lora_alpha=16, 
        target_modules="all-linear", # Memasang chip ke seluruh lapisan Linear
        bias="none"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print("✅ LORA BERHASIL DIINJEKSI! BEBAN VRAM SEKARANG SANGAT RINGAN.")
except Exception as e:
    print("⚠️ LORA GAGAL:", e)
print("="*60 + "\\n")
"""

    # Taktik Injeksi: Menyisipkan LoRA tepat sebelum Optimizer dibuat, dengan menjaga struktur indentasi
    content = re.sub(
        r'([ \t]+)([a-zA-Z0-9_]+)\s*=\s*torch\.optim\.AdamW\(',
        lambda m: m.group(1) + lora_code.replace('\n', '\n' + m.group(1)) + '\n' + m.group(0),
        content
    )

    with open(script_path, 'w') as f:
        f.write(content)
        
    print("✅ INJEKSI LORA SUKSES! Model VLA-mu kini versi taktis yang super ringan.")
else:
    print("❌ ERROR: Skrip tidak ditemukan di Kaggle!")

📦 Mengunduh perlengkapan operasi LoRA (PEFT)...
🧹 Melakukan Factory Reset pada skrip untuk membuang patch lama...
✅ INJEKSI LORA SUKSES! Model VLA-mu kini versi taktis yang super ringan.


Updated 1 path from the index


## 9️. !!! KHUSUS Restore Checkpoint & Force Resume !!!

**Untuk melanjutkan training dari step 1500:** Restore checkpoint dari /kaggle/input/ dan set ulang internal step counter ke 1500. True Overwrite memastikan file selalu ditimpa ke folder "LATEST".

In [9]:
# ============================================================
# CELL ULTIMATE: RESTORE 1500 + TRUE OVERWRITE + FORCE RESUME
# ============================================================
import os, shutil, re, textwrap

print("🚀 MEMULAI PERSIAPAN MUTLAK SESI BARU (START 1500)...")

# 1. PEMULIHAN LOGISTIK (AUTO-DETECT & RESTORE)
target_dir = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
source_dir = '/kaggle/input/' # Pelacak otomatis Kaggle Input

print("🔍 1. Mencari file checkpoint evakuasi 1500-mu...")
found_dir = None
for root, dirs, files in os.walk(source_dir):
    if 'model.safetensors' in files:
        found_dir = root
        break

if found_dir:
    print(f"📦 Mengamankan memori dari: {found_dir}")
    os.makedirs(target_dir, exist_ok=True)
    for item in os.listdir(found_dir):
        s = os.path.join(found_dir, item)
        d = os.path.join(target_dir, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
        else: shutil.copy2(s, d)
    print("✅ Restore Berhasil! File diletakkan di folder absolut 'LATEST'.")
else:
    print("⚠️ Checkpoint tidak ditemukan! Pastikan kamu sudah Add Input Dataset Kaggle 1500!")

# 2. PATCH SCRIPT: TRUE OVERWRITE & FORCE RESUME
print("\n🛠️ 2. Menyuntikkan True Overwrite & Force Resume...")
script_path = '/kaggle/working/openpi/scripts/train_pytorch.py'

if os.path.exists(script_path):
    with open(script_path, 'r') as f:
        content = f.read()

    # A. Patch True Overwrite (Memaksa PyTorch menimpa file yang sama persis)
    overwrite_code = """
# --- TRUE OVERWRITE HIJACK ---
import os, safetensors.torch
STATIC_DIR = "/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST"
os.makedirs(STATIC_DIR, exist_ok=True)

_orig_save_model = safetensors.torch.save_model
def _hijacked_save_model(model, filename, *args, **kwargs):
    target_file = os.path.join(STATIC_DIR, "model.safetensors")
    print(f"\\n♻️ TRUE OVERWRITE: Menimpa file {target_file}")
    _orig_save_model(model, target_file, *args, **kwargs)
safetensors.torch.save_model = _hijacked_save_model

_orig_torch_save = torch.save
def _hijacked_torch_save(obj, f, *args, **kwargs):
    if "optimizer.pt" in str(f):
        target_file = os.path.join(STATIC_DIR, "optimizer.pt")
        print(f"♻️ TRUE OVERWRITE: Menimpa file {target_file}")
        return _orig_torch_save(obj, target_file, *args, **kwargs)
    return _orig_torch_save(obj, f, *args, **kwargs)
torch.save = _hijacked_torch_save
# -----------------------------
"""
    if "# --- TRUE OVERWRITE HIJACK ---" not in content:
        if "import torch" not in content[:500]:
            content = "import torch\n" + overwrite_code + content
        else:
            content = content.replace("import torch", "import torch\n" + overwrite_code, 1)

    # B. Patch Force Resume (Memaksa AI menelan data dari folder LATEST & Start 1500)
    resume_code = textwrap.dedent("""
        # --- FORCE RESUME 1500 ---
        from safetensors.torch import load_file
        if os.path.exists(os.path.join(STATIC_DIR, "model.safetensors")):
            print("\\n" + "🔥"*25)
            print("🚀 FORCE RESUME DARI FOLDER 'LATEST' AKTIF!")
            try:
                model.load_state_dict(load_file(os.path.join(STATIC_DIR, "model.safetensors")), strict=False)
                print("✅ Bobot AI berhasil dimuat!")
            except Exception as e: print(f"❌ Error bobot: {e}")
            try:
                if os.path.exists(os.path.join(STATIC_DIR, "optimizer.pt")):
                    optim.load_state_dict(torch.load(os.path.join(STATIC_DIR, "optimizer.pt"), map_location="cpu", weights_only=False))
                    print("✅ Optimizer berhasil dimuat!")
            except Exception as e: print(f"❌ Error optimizer: {e}")
            
            # JAM INTERNAL DIPUTAR KE 1500
            global_step = 1500
        # -------------------------
    """)
    
    match = re.search(r'^([ \t]*)model\.train\(\)', content, flags=re.MULTILINE)
    if match and "# --- FORCE RESUME 1500 ---" not in content:
        indent = match.group(1)
        indented_code = "".join(indent + line + "\n" for line in resume_code.strip().split('\n'))
        content = content.replace(match.group(0), match.group(0) + "\n" + indented_code, 1)

    with open(script_path, 'w') as f:
        f.write(content)
    print("✅ Patch Selesai! True Overwrite & Resume siap bertugas.")
else:
    print("❌ ERROR: Skrip train_pytorch.py tidak ditemukan.")

🚀 MEMULAI PERSIAPAN MUTLAK SESI BARU (START 1500)...
🔍 1. Mencari file checkpoint evakuasi 1500-mu...
📦 Mengamankan memori dari: /kaggle/input/datasets/sjankaczar/dronepivla/checkpoint_1500_rescue
✅ Restore Berhasil! File diletakkan di folder absolut 'LATEST'.

🛠️ 2. Menyuntikkan True Overwrite & Force Resume...
✅ Patch Selesai! True Overwrite & Resume siap bertugas.


## 10. Launch Training

**Eksekusi training di GPU T4 Kaggle dengan konfigurasi stabil.** Monitor VRAM dan catat output training.

In [10]:
# ============================================================
# Cell 7: Launch Training (Kaggle SINGLE GPU - STABLE MODE)
# ============================================================
import sys, os, subprocess, signal, shutil, torch

# Bersihkan cache VRAM sebelum mulai
torch.cuda.empty_cache()

# 1. Konfigurasi Environment (Kaggle T4 x RAM 30GB)
env = os.environ.copy()
env['PYTHONPATH'] = "/kaggle/working/openpi/src:/kaggle/working/openpi/packages/openpi-client/src"
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_OFFLINE'] = '1'  

env['JAX_PLATFORMS'] = 'cpu' 
env['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
env['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

# Memaksimalkan efisiensi memori VRAM GPU
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# --- KONFIGURASI AMAN ---
# Kita gunakan Batch Size 1 agar VRAM GPU 15GB tidak penuh saat Adam Optimizer bekerja.
# Kita JANGAN pakai Dual GPU agar RAM Sistem 30GB tidak meledak (SIGKILL / Exit -9).
STABLE_BATCH_SIZE = 1

# 2. Build Command (Single GPU Standar)
cmd = [
    sys.executable,
    '/kaggle/working/openpi/scripts/train_pytorch.py',
    CONFIG_NAME, 
    f'--exp_name={EXP_NAME}',
    f'--batch_size={STABLE_BATCH_SIZE}',
    f'--num_train_steps={NUM_STEPS}',
    f'--save_interval={SAVE_INTERVAL}',
    f'--log_interval={LOG_INTERVAL}',
]

if not USE_WANDB:
    cmd.append('--no-wandb-enabled')

print('🚀 MEMULAI TRAINING DI KAGGLE (STABLE SINGLE GPU)...')
print(f'Batch Size: {STABLE_BATCH_SIZE} | Menghindari SIGKILL RAM 30GB')
print(f'Perintah: {" ".join(cmd)}')
print('=' * 80)

# 3. Jalankan Training
process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/kaggle/working',
)

try:
    for line in process.stdout:
        if "Unable to initialize backend" not in line:
            print(line, end='')
    process.wait()
except KeyboardInterrupt:
    print('\n⚠️ Training dihentikan paksa oleh user.')
    process.send_signal(signal.SIGINT)
    process.wait(timeout=30)

# 4. Pengumpulan Hasil
if process.returncode == 0:
    print('\n' + '=' * 80)
    print('🎉 TRAINING SELESAI DENGAN SUKSES DI KAGGLE!')

    FINAL_OUTPUT_DIR = f'/kaggle/working/final_model_{EXP_NAME}'
    LOCAL_CHECKPOINT = f'/kaggle/working/checkpoints/{CONFIG_NAME}/{EXP_NAME}'

    if os.path.exists(LOCAL_CHECKPOINT):
        if os.path.exists(FINAL_OUTPUT_DIR): shutil.rmtree(FINAL_OUTPUT_DIR)
        shutil.copytree(LOCAL_CHECKPOINT, FINAL_OUTPUT_DIR)

        ns_src = '/kaggle/working/assets/pi0_drone_lite/rafli/drone_roblox/norm_stats.json'
        if os.path.exists(ns_src):
            shutil.copy2(ns_src, os.path.join(FINAL_OUTPUT_DIR, 'norm_stats.json'))
            print("✅ Stats normalisasi disertakan.")

        print(f"\n✨ SEMUA HASIL SUDAH SIAP!")
    else:
        print(f"❌ ERROR: Checkpoint tidak ditemukan di {LOCAL_CHECKPOINT}")

else:
    print(f'\n⚠️ Training berhenti dengan kode error {process.returncode}')

🚀 MEMULAI TRAINING DI KAGGLE (STABLE SINGLE GPU)...
Batch Size: 1 | Menghindari SIGKILL RAM 30GB
Perintah: /usr/bin/python3 /kaggle/working/openpi/scripts/train_pytorch.py pi0_drone_lite --exp_name=drone_gimbal_v1 --batch_size=1 --num_train_steps=3000 --save_interval=500 --log_interval=10 --no-wandb-enabled
2026-04-10 01:28:52.171141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775784532.193149     417 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775784532.199773     417 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775784532.216833     417 computation_placer.cc:177] computation placer already registered. Please check linkage and

## 11. Monitor Checkpoint Status

Memantau file checkpoint yang sudah disimpan dan waktu update terakhirnya.

In [11]:
# ============================================================
# Cell Radar: PEMANTAU STATUS PENYIMPANAN CHECKPOINT
# ============================================================
import os, time

target_dir = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
print("📡 MEMINDAI GUDANG PENYIMPANAN 'LATEST'...")

if os.path.exists(target_dir):
    files = os.listdir(target_dir)
    if not files:
        print("⚠️ Folder ada, tapi isinya kosong.")
    else:
        for f in files:
            file_path = os.path.join(target_dir, f)
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            # Mengecek jam berapa file ini terakhir diperbarui/ditimpa
            mod_time = time.ctime(os.path.getmtime(file_path)) 
            print(f"📄 {f} | Ukuran: {size_mb:.2f} MB | Terakhir Diperbarui: {mod_time}")
else:
    print("🚧 Gudang belum terbentuk. AI belum mencapai jadwal save (misal: Step 2000).")

📡 MEMINDAI GUDANG PENYIMPANAN 'LATEST'...
📄 metadata.pt | Ukuran: 0.00 MB | Terakhir Diperbarui: Fri Apr 10 01:12:42 2026
📄 model.safetensors | Ukuran: 6770.25 MB | Terakhir Diperbarui: Fri Apr 10 03:44:51 2026
📄 assets | Ukuran: 0.00 MB | Terakhir Diperbarui: Fri Apr 10 01:12:41 2026
📄 optimizer.pt | Ukuran: 132.17 MB | Terakhir Diperbarui: Fri Apr 10 03:44:52 2026


## 1️2. Zip & Download Checkpoint

Mengompresi folder LATEST menjadi ZIP untuk didownload dari Kaggle.

In [12]:
import shutil
from IPython.display import FileLink, display

folder_latest = '/kaggle/working/checkpoints/pi0_drone_lite/drone_gimbal_v1/LATEST'
zip_path = '/kaggle/working/checkpoint_LATEST_rescue'

print("📦 Membungkus folder LATEST menjadi ZIP (Mohon tunggu)...")
shutil.make_archive(zip_path, 'zip', folder_latest)

print("\n✅ KOMPRESI BERHASIL! KLIK LINK BIRU INI UNTUK DOWNLOAD:")
display(FileLink('checkpoint_LATEST_rescue.zip'))

📦 Membungkus folder LATEST menjadi ZIP (Mohon tunggu)...

✅ KOMPRESI BERHASIL! KLIK LINK BIRU INI UNTUK DOWNLOAD:


/kaggle/working/checkpoint_LATEST_rescue.zip

## 1️3. Test Drive Model - Inference Only

**Mengujicoba model yang sudah dilatih untuk inference tanpa training.** Memuat checkpoint, memberikan image + prompt, dan melihat action prediction dari VLA model.

In [ ]:
# ============================================================
# 🎯 OPERASI KHUSUS: TEST DRIVE MODEL (INFERENCE ONLY)
# ============================================================
import os, shutil, torch, gc
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from safetensors.torch import load_file, save_file
from datasets import load_dataset

print("🚀 MEMULAI OPERASI UJI COBA (TEST DRIVE)...")

# 1. PENCARIAN & PERSIAPAN OTAK AI (CHECKPOINT)
search_dir = '/kaggle/input/'
test_dir = '/kaggle/working/test_checkpoint'
dirty_path = os.path.join(test_dir, "model.safetensors")
clean_path = os.path.join(test_dir, "model_clean.safetensors")

print("🔍 1. Melacak keberadaan otak AI di gudang Kaggle...")
found_dir = None
for root, dirs, files in os.walk(search_dir):
    if 'model.safetensors' in files:
        found_dir = root
        break

if found_dir and not os.path.exists(test_dir):
    print(f"📦 Mengangkut otak AI dari {found_dir} ke ruang uji coba...")
    os.makedirs(test_dir, exist_ok=True)
    for item in os.listdir(found_dir):
        s = os.path.join(found_dir, item)
        d = os.path.join(test_dir, item)
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
        else: shutil.copy2(s, d)
    print("✅ Otak AI berhasil dipasang di ruang uji!")
elif os.path.exists(test_dir):
    print("✅ Otak AI sudah ada di ruang uji.")
else:
    raise FileNotFoundError("❌ ERROR: Checkpoint tidak ditemukan! Pastikan sudah Add Input.")

# 2. MEMBERSIHKAN BOBOT LORA (PENTING UNTUK INFERENSI)
if not os.path.exists(clean_path):
    print("🛠️ 2. Menyeterika baju zirah LoRA agar bisa dibaca mesin standar...")
    state_dict = load_file(dirty_path)
    clean_dict = {k.replace("base_model.model.", ""): v for k, v in state_dict.items()}
    save_file(clean_dict, clean_path)
    print("✅ Bobot AI berhasil dibersihkan!")

# 3. PERSIAPAN MATA AI (GAMBAR & PROMPT)
print("\n🖼️ 3. Mengambil gambar dari Dataset Roblox...")
img = None
try:
    ds = load_dataset("rafli/drone_roblox", split="train")
    img_col = 'image' if 'image' in ds.column_names else ds.column_names[0]
    img = ds[0][img_col] # Mengambil gambar pertama
except:
    print("⚠️ Gagal load dataset. Menggunakan gambar darurat...")
    img = Image.new('RGB', (224, 224), color='#87CEEB')
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 112, 224, 224], fill='#A9A9A9')
    draw.rectangle([90, 80, 134, 124], fill='red')

if img.mode != 'RGB': img = img.convert('RGB')

# 🔥 MASUKKAN PERINTAHMU UNTUK DRONE DI SINI 🔥
PROMPT_TEKS = "terbang maju menuju kubus merah" 

plt.imshow(img)
plt.title(f"Prompt: '{PROMPT_TEKS}'")
plt.axis('off')
plt.show()

# 4. EKSEKUSI PREDIKSI!
print("\n🧠 4. Membangkitkan Mesin Inferensi VLA...")
try:
    from openpi.policies import policy_factory
    
    # Tukar file sementara agar policy_factory membaca file yang sudah bersih
    os.rename(dirty_path, os.path.join(test_dir, "model_dirty.bak"))
    os.rename(clean_path, dirty_path)
    
    policy = policy_factory.create_policy(
        config_name="pi0_drone_lite",
        checkpoint_dir=test_dir
    )
    
    print("⚙️ AI SANGAT FOKUS... MEMPREDIKSI AKSI BERDASARKAN GAMBAR & TEKS...")
    action_result = policy({"image": img, "prompt": PROMPT_TEKS})
    
    print("\n" + "🔥"*25)
    print("🎯 HASIL PREDIKSI (ACTION TENSOR):")
    print(action_result)
    print("🔥"*25 + "\n")
    
except Exception as e:
    print(f"❌ Error saat inferensi: {e}")
finally:
    # Kembalikan file seperti semula agar tidak error di pengujian selanjutnya
    if os.path.exists(os.path.join(test_dir, "model_dirty.bak")):
        if os.path.exists(dirty_path): os.rename(dirty_path, clean_path)
        os.rename(os.path.join(test_dir, "model_dirty.bak"), dirty_path)